# rvt-rs — Python quickstart

This notebook walks through the most common things you'll want to do with the Python bindings: open a Revit file, inspect its metadata, read the root document via the schema-directed walker, and export to IFC4.

**Prereqs**: `pip install rvt` (wheel available once #52e ships) or `maturin build --release --features python` + `pip install target/wheels/rvt-*.whl` from a source checkout.

**Sample file**: this notebook uses `racbasicsamplefamily-2024.rfa` from the [phi-ag/rvt](https://github.com/phi-ag/rvt) fixture corpus. Adjust the path in the cell below to point at any Revit file you have locally.

In [ ]:
import rvt
print(f"rvt version: {rvt.__version__}")

SAMPLE = "../../samples/racbasicsamplefamily-2024.rfa"  # adjust as needed

## 1. Open the file and inspect metadata

Constructing a `RevitFile` opens the OLE2 / CFB container and caches the file bytes. Property accesses (`.version`, `.part_atom_title`, etc.) parse the relevant stream lazily; nothing costly happens until you call a method that actually reads bytes.

In [ ]:
f = rvt.RevitFile(SAMPLE)

print(f"Revit release     : {f.version}")
print(f"Build tag         : {f.build}")
print(f"Document title    : {f.part_atom_title}")
print(f"Document GUID     : {f.guid}")
print(f"Original path     : {f.original_path}")  # from the creator's machine

## 2. List OLE streams

A Revit file is a Microsoft Compound File Binary (MS-CFB) container holding a set of named streams. Every release 2016–2026 contains the same 12 invariant streams plus a version-specific `Partitions/NN`.

In [ ]:
for s in f.stream_names():
    print(f"  {s}")

missing = f.missing_required_streams()
if missing:
    print(f"\n⚠️  missing required streams: {missing}")
else:
    print("\n✅ all required streams present")

## 3. Schema inventory

The `Formats/Latest` stream carries Revit's complete serialization schema: every class name, every field, every type. `rvt-rs` classifies **100% of all 13,570 fields** across all 11 supported releases (Q5.2 result).

In [ ]:
summary = f.schema_summary()
print(f"Classes in this file      : {summary['classes']}")
print(f"Total declared fields     : {summary['fields']}")
print(f"Distinct C++ type sigs    : {summary['cpp_types']}")

## 4. Read the root ADocument record

`read_adocument()` runs the Layer-5a walker and returns ADocument's 13 declared fields as typed values. This is the first layer of actual instance-data extraction (as opposed to just schema).

The walker has been cross-version-validated: releases 2024–2026 all produce byte-identical ElementId values for the last three reference fields (`m_ownerFamilyId`, `m_ownerFamilyContainingGroupId`, `m_devBranchInfo`).

In [ ]:
doc = f.read_adocument()
if doc is None:
    print("walker couldn't locate ADocument — file layout unsupported")
else:
    print(f"ADocument at offset 0x{doc['entry_offset']:06x}  (Revit {doc['version']})")
    print()
    for i, field in enumerate(doc['fields']):
        kind = field['kind']
        name = field['name']
        detail = ''
        if kind == 'pointer':
            detail = f"slots [{field['slot_a']:#x}, {field['slot_b']:#x}]"
        elif kind == 'element_id':
            detail = f"tag={field['tag']}, id={field['id']}"
        elif kind == 'ref_container':
            detail = f"count={field['count']}"
        print(f"  [{i:>2}] {name:<36} {kind:<14}  {detail}")

## 5. Export to IFC4

`write_ifc()` produces a spec-valid IFC4 STEP text: ISO-10303-21 envelope, the required framework entities (IfcPerson / IfcOrganization / IfcApplication / IfcOwnerHistory / IfcSIUnit / IfcUnitAssignment / IfcGeometricRepresentationContext), and an IfcProject populated from the document metadata.

Current coverage is document-level. Building-element geometry is pending walker expansion — see task #52 in the repo for the roadmap.

In [ ]:
ifc = f.write_ifc()
print(f"IFC output length    : {len(ifc):,} bytes")
print(f"IFC4 schema declared : {'FILE_SCHEMA((' in ifc and 'IFC4' in ifc}")
print(f"IFCPROJECT present    : {ifc.count('IFCPROJECT(')} entity")

# Save it next to the input.
out_path = SAMPLE.replace('.rfa', '.ifc')
with open(out_path, 'w') as fh:
    fh.write(ifc)
print(f"Wrote {out_path}")

# Preview the first 20 lines.
print()
for line in ifc.splitlines()[:20]:
    print(f"  {line}")

## 6. Sanity check: cross-version consistency

If you have multiple releases of the same document, the walker's output should be stable within each "version band". Here we just re-export to IFC and compare project names — but you could extend this to compare `read_adocument()` outputs.

(Skip this cell if you only have the one sample.)

In [ ]:
import pathlib

CORPUS = pathlib.Path(SAMPLE).parent
available = sorted(CORPUS.glob('*.rfa'))

for path in available:
    try:
        g = rvt.RevitFile(str(path))
        print(f"  {path.name:<45} Revit {g.version}  title={g.part_atom_title!r}")
    except Exception as exc:
        print(f"  {path.name:<45} <failed: {exc}>")

## Where to next?

- **Recon report**: `docs/rvt-moat-break-reconnaissance.md` — the living technical record of everything we know about the Revit on-disk format, from container layer through the Layer-5a walker.
- **API docs**: [docs/python.md](python.md) for the full Python API reference.
- **Contributing**: [CONTRIBUTING.md](../CONTRIBUTING.md), especially the *Where help is most wanted* section if you want to extend the walker to cover more classes.
- **Issues**: [github.com/DrunkOnJava/rvt-rs/issues](https://github.com/DrunkOnJava/rvt-rs/issues)